# The command-line tools

wfdb-x includes all the command-line tools of the WFDB package, such as
`rdsamp`, `rdann`, `wfdbdesc`, `ann2rr`, `xform`, `snip` and `mit2edf`. In a
shell, run them through the `wfdbx` command:

```
export WFDBX_INPUT_DIR=$(python -c "import wfdbx; print(wfdbx.sample_data_dir())")
wfdbx rdsamp -r 100s -t 0:5 -v -p
wfdbx --list
```

In this notebook, the tools run through `subprocess`. They read records from
the input directory, given by the `WFDBX_INPUT_DIR` environment variable,
and write to the output directory, given by `WFDBX_OUTPUT_DIR`. Both are the
current directory by default. The helper below runs a tool with the
package's data directory, `wfdbx.sample_data_dir()`, as its input directory.
It holds the sample record `100s`, included with wfdb-x: the first minute of
record 100 of the MIT-BIH Arrhythmia Database.

In [ ]:
import os
import subprocess
import sys

import wfdbx



def tool(*args, stdin=None, input_dir=wfdbx.sample_data_dir()):
    """Run a wfdb-x tool, reading records under input_dir, and print its
    output."""
    env = dict(os.environ, WFDBX_INPUT_DIR=input_dir)
    result = subprocess.run([sys.executable, "-m", "wfdbx.cli", *args],
                            input=stdin, capture_output=True, text=True,
                            env=env)
    print(result.stdout + result.stderr, end="")
    return result.returncode

In [ ]:
tool("wfdbdesc", "100s")

`rdsamp` prints samples. `-p` prints them in physical units, and `-v` adds
column headings. Units that the header does not give are shown as `nd`
("not defined").

In [ ]:
tool("rdsamp", "-r", "100s", "-t", "s5", "-v", "-p")

In [ ]:
tool("rdann", "-r", "100s", "-a", "atr", "-t", "0:3")

`ann2rr` prints RR intervals. `-i s` prints them in seconds:

In [ ]:
tool("ann2rr", "-r", "100s", "-a", "atr", "-i", "s", "-t", "0:5")

## Writing a record from text

`wrsamp` writes a record from columns of numbers. This example writes a
record, `fromtext`, in the current directory. You must give the sampling
frequency (`-F`), the gain (`-G`), the ADC resolution (`-R`) and the units
(`-U`):

In [ ]:
text = "".join(f"{i} {i % 7 - 3}\n" for i in range(20))
tool("wrsamp", "-o", "fromtext", "-F", "100", "-G", "200", "-R", "16",
     "-U", "mV", stdin=text)
# (wrsamp wrote the record in the current directory)
tool("wfdbdesc", "fromtext", input_dir=".")

In [ ]:
# Without -F, wrsamp stops with an error (status 1):
tool("wrsamp", "-o", "fromtext", stdin=text)